# Underspec Task Categorization Analysis

This notebook analyzes the results of categorizing underspecified tasks into:
1. **Originally Overspecified** - passes at pass@3 both before and after underspecification
2. **New Task** - doesn't pass@3 after underspec, but judge thinks task is complete
3. **Underspecified** - doesn't pass@3 after underspec, and judge agrees task is incomplete

In [63]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML, Markdown
from collections import Counter

In [64]:
# Configuration - update this path as needed
DATASET_DIR = Path("../experiments/underspec_datasets/intersection_delete_v1_20260120_110000")
DATASET_DIR = Path("/Users/sam.denton/Code/models/research/lhaw/experiments/mcpatlas/experiments/mcpatlas/underspec_datasets/intersection_delete_2segments_20260121_161230")

# Load both categorization results
with open(DATASET_DIR / "categorization_completion.json") as f:
    completion_results = json.load(f)

with open(DATASET_DIR / "categorization_consistency.json") as f:
    consistency_results = json.load(f)

print("Loaded results from:", DATASET_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sam.denton/Code/models/research/lhaw/experiments/mcpatlas/experiments/mcpatlas/underspec_datasets/intersection_delete_2segments_20260121_161230/categorization_completion.json'

## Summary Comparison

In [56]:
def create_summary_visualization(completion_results, consistency_results):
    """Create a beautiful HTML summary comparison."""
    c_sum = completion_results["summary"]
    s_sum = consistency_results["summary"]
    total = c_sum["total_tasks"]
    
    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <!-- Header -->
        <div style="text-align: center; margin-bottom: 24px;">
            <h2 style="margin: 0; color: #1a237e;">Categorization Summary</h2>
            <p style="color: #666; margin: 8px 0 0 0;">{total} total tasks analyzed</p>
        </div>
        
        <!-- Cards Row -->
        <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; margin-bottom: 24px;">
            <!-- Originally Overspecified -->
            <div style="background: linear-gradient(135deg, #e8f5e9 0%, #c8e6c9 100%); border-radius: 12px; padding: 20px; border: 2px solid #4caf50;">
                <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 1px; color: #2e7d32; margin-bottom: 8px;">Originally Overspecified</div>
                <div style="font-size: 36px; font-weight: 700; color: #1b5e20;">{c_sum['originally_overspecified']}</div>
                <div style="font-size: 14px; color: #388e3c;">{100*c_sum['originally_overspecified']/total:.1f}% of tasks</div>
                <div style="font-size: 12px; color: #666; margin-top: 8px;">Still pass after underspec</div>
            </div>
            
            <!-- New Tasks -->
            <div style="background: linear-gradient(135deg, #fff3e0 0%, #ffe0b2 100%); border-radius: 12px; padding: 20px; border: 2px solid #ff9800;">
                <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 1px; color: #e65100; margin-bottom: 8px;">New Tasks</div>
                <div style="display: flex; gap: 16px; align-items: baseline;">
                    <div>
                        <div style="font-size: 36px; font-weight: 700; color: #e65100;">{c_sum['new_tasks']}</div>
                        <div style="font-size: 11px; color: #f57c00;">Completion</div>
                    </div>
                    <div style="font-size: 24px; color: #999;">vs</div>
                    <div>
                        <div style="font-size: 36px; font-weight: 700; color: #e65100;">{s_sum['new_tasks']}</div>
                        <div style="font-size: 11px; color: #f57c00;">Consistency</div>
                    </div>
                </div>
                <div style="font-size: 12px; color: #666; margin-top: 8px;">Complete despite fail</div>
            </div>
            
            <!-- Underspecified -->
            <div style="background: linear-gradient(135deg, #ffebee 0%, #ffcdd2 100%); border-radius: 12px; padding: 20px; border: 2px solid #f44336;">
                <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 1px; color: #c62828; margin-bottom: 8px;">Truly Underspecified</div>
                <div style="display: flex; gap: 16px; align-items: baseline;">
                    <div>
                        <div style="font-size: 36px; font-weight: 700; color: #c62828;">{c_sum['underspecified']}</div>
                        <div style="font-size: 11px; color: #d32f2f;">Completion</div>
                    </div>
                    <div style="font-size: 24px; color: #999;">vs</div>
                    <div>
                        <div style="font-size: 36px; font-weight: 700; color: #c62828;">{s_sum['underspecified']}</div>
                        <div style="font-size: 11px; color: #d32f2f;">Consistency</div>
                    </div>
                </div>
                <div style="font-size: 12px; color: #666; margin-top: 8px;">Genuinely ambiguous</div>
            </div>
        </div>
        
        <!-- Detailed Comparison Table -->
        <div style="background: white; border-radius: 12px; border: 1px solid #e0e0e0; overflow: hidden;">
            <table style="width: 100%; border-collapse: collapse;">
                <thead>
                    <tr style="background: #f5f5f5;">
                        <th style="padding: 12px 16px; text-align: left; font-weight: 600; color: #333; border-bottom: 2px solid #e0e0e0;">Metric</th>
                        <th style="padding: 12px 16px; text-align: center; font-weight: 600; color: #1976d2; border-bottom: 2px solid #e0e0e0;">Completion Method</th>
                        <th style="padding: 12px 16px; text-align: center; font-weight: 600; color: #7b1fa2; border-bottom: 2px solid #e0e0e0;">Consistency Method</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td style="padding: 12px 16px; border-bottom: 1px solid #f0f0f0;">Originally Overspecified</td>
                        <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #f0f0f0;">{c_sum['originally_overspecified']} ({100*c_sum['originally_overspecified']/total:.1f}%)</td>
                        <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #f0f0f0;">{s_sum['originally_overspecified']} ({100*s_sum['originally_overspecified']/total:.1f}%)</td>
                    </tr>
                    <tr style="background: #fafafa;">
                        <td style="padding: 12px 16px; border-bottom: 1px solid #f0f0f0;">New Tasks</td>
                        <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #f0f0f0; color: #e65100; font-weight: 500;">{c_sum['new_tasks']} ({100*c_sum['new_tasks']/total:.1f}%)</td>
                        <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #f0f0f0; color: #e65100; font-weight: 500;">{s_sum['new_tasks']} ({100*s_sum['new_tasks']/total:.1f}%)</td>
                    </tr>
                    <tr>
                        <td style="padding: 12px 16px; border-bottom: 1px solid #f0f0f0;">Underspecified</td>
                        <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #f0f0f0; color: #c62828; font-weight: 500;">{c_sum['underspecified']} ({100*c_sum['underspecified']/total:.1f}%)</td>
                        <td style="padding: 12px 16px; text-align: center; border-bottom: 1px solid #f0f0f0; color: #c62828; font-weight: 500;">{s_sum['underspecified']} ({100*s_sum['underspecified']/total:.1f}%)</td>
                    </tr>
                    <tr style="background: #fafafa;">
                        <td style="padding: 12px 16px;">Judge Model</td>
                        <td style="padding: 12px 16px; text-align: center; font-family: monospace; font-size: 12px;">{c_sum.get('judge_model', 'N/A')}</td>
                        <td style="padding: 12px 16px; text-align: center; font-family: monospace; font-size: 12px;">{s_sum.get('judge_model', 'N/A')}</td>
                    </tr>
                </tbody>
            </table>
        </div>
    </div>
    """
    display(HTML(html))

create_summary_visualization(completion_results, consistency_results)

Metric,Completion Method,Consistency Method
Originally Overspecified,27 (38.6%),27 (38.6%)
New Tasks,33 (47.1%),16 (22.9%)
Underspecified,10 (14.3%),27 (38.6%)
Judge Model,gemini/gemini-3-flash-preview,gemini/gemini-3-flash-preview


## Agreement Between Methods

In [57]:
# Find tasks where methods disagree (only for judged tasks)
completion_new = set(completion_results["new_tasks"])
completion_underspec = set(completion_results["underspecified"])
consistency_new = set(consistency_results["new_tasks"])
consistency_underspec = set(consistency_results["underspecified"])

# Agreement/disagreement
both_new = completion_new & consistency_new
both_underspec = completion_underspec & consistency_underspec
completion_new_consistency_underspec = completion_new & consistency_underspec
completion_underspec_consistency_new = completion_underspec & consistency_new

total_judged = len(completion_new) + len(completion_underspec)
total_agree = len(both_new) + len(both_underspec)

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
    <h3 style="color: #333; margin-bottom: 16px;">Agreement Analysis</h3>
    <p style="color: #666; margin-bottom: 20px;">Comparing how the two methods classified the {total_judged} tasks that failed pass@3</p>
    
    <!-- Agreement Stats -->
    <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px; margin-bottom: 24px;">
        <div style="background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%); border-radius: 12px; padding: 20px; text-align: center; border: 2px solid #2196f3;">
            <div style="font-size: 48px; font-weight: 700; color: #1565c0;">{total_agree}</div>
            <div style="font-size: 14px; color: #1976d2; font-weight: 500;">Tasks Agree</div>
            <div style="font-size: 24px; color: #1976d2; margin-top: 4px;">{100*total_agree/total_judged:.1f}%</div>
        </div>
        <div style="background: linear-gradient(135deg, #fce4ec 0%, #f8bbd9 100%); border-radius: 12px; padding: 20px; text-align: center; border: 2px solid #e91e63;">
            <div style="font-size: 48px; font-weight: 700; color: #c2185b;">{total_judged - total_agree}</div>
            <div style="font-size: 14px; color: #d81b60; font-weight: 500;">Tasks Disagree</div>
            <div style="font-size: 24px; color: #d81b60; margin-top: 4px;">{100*(total_judged - total_agree)/total_judged:.1f}%</div>
        </div>
    </div>
    
    <!-- Confusion Matrix Style -->
    <div style="background: white; border-radius: 12px; border: 1px solid #e0e0e0; overflow: hidden;">
        <div style="background: #f5f5f5; padding: 12px 16px; font-weight: 600; color: #333; text-align: center; border-bottom: 1px solid #e0e0e0;">
            Agreement Matrix
        </div>
        <table style="width: 100%; border-collapse: collapse;">
            <thead>
                <tr>
                    <th style="padding: 12px; background: #fafafa; border-bottom: 1px solid #e0e0e0;"></th>
                    <th style="padding: 12px; background: #e8f5e9; color: #2e7d32; text-align: center; border-bottom: 1px solid #e0e0e0;" colspan="2">Consistency Method</th>
                </tr>
                <tr>
                    <th style="padding: 12px; background: #fafafa;"></th>
                    <th style="padding: 12px; background: #fff3e0; color: #e65100; text-align: center; font-size: 13px;">New Task</th>
                    <th style="padding: 12px; background: #ffebee; color: #c62828; text-align: center; font-size: 13px;">Underspecified</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <th style="padding: 12px; background: #fff3e0; color: #e65100; text-align: right; font-size: 13px; writing-mode: horizontal-tb;">Completion: New Task</th>
                    <td style="padding: 16px; text-align: center; background: #c8e6c9; font-size: 24px; font-weight: 700; color: #2e7d32;">{len(both_new)}</td>
                    <td style="padding: 16px; text-align: center; background: #ffcdd2; font-size: 24px; font-weight: 700; color: #c62828;">{len(completion_new_consistency_underspec)}</td>
                </tr>
                <tr>
                    <th style="padding: 12px; background: #ffebee; color: #c62828; text-align: right; font-size: 13px;">Completion: Underspecified</th>
                    <td style="padding: 16px; text-align: center; background: #ffcdd2; font-size: 24px; font-weight: 700; color: #c62828;">{len(completion_underspec_consistency_new)}</td>
                    <td style="padding: 16px; text-align: center; background: #c8e6c9; font-size: 24px; font-weight: 700; color: #2e7d32;">{len(both_underspec)}</td>
                </tr>
            </tbody>
        </table>
        <div style="padding: 12px 16px; background: #fafafa; font-size: 12px; color: #666; border-top: 1px solid #e0e0e0;">
            <span style="color: #2e7d32;">■</span> Green = Agreement &nbsp;&nbsp;
            <span style="color: #c62828;">■</span> Red = Disagreement
        </div>
    </div>
</div>
"""
display(HTML(html))

## Tasks Where Methods Disagree

In [58]:
def get_category_style(category):
    """Get color styling based on category."""
    styles = {
        "originally_overspecified": {"bg": "#e8f5e9", "border": "#4caf50", "badge": "#2e7d32"},
        "new_task": {"bg": "#fff3e0", "border": "#ff9800", "badge": "#e65100"},
        "underspecified": {"bg": "#ffebee", "border": "#f44336", "badge": "#c62828"},
    }
    return styles.get(category, {"bg": "#f5f5f5", "border": "#9e9e9e", "badge": "#616161"})

def display_task(task_id, completion_details, consistency_details):
    """Display a single task's details from both methods with improved styling."""
    c_det = completion_details.get(task_id, {})
    s_det = consistency_details.get(task_id, {})
    
    c_style = get_category_style(c_det.get('category'))
    s_style = get_category_style(s_det.get('category'))
    
    # Format removed segments in black text with subtle styling
    segments = c_det.get('removed_segments', [])
    segments_html = " ".join([f'<span style="background: #f5f5f5; border: 1px solid #ddd; padding: 4px 10px; border-radius: 4px; margin: 2px; display: inline-block; font-size: 13px; color: #111; font-weight: 500;">{s}</span>' for s in segments])
    
    # Get prompts
    original_prompt = c_det.get('original_prompt', 'N/A')
    underspec_prompt = c_det.get('underspec_prompt', 'N/A')
    
    # Truncate reasoning smartly
    c_reasoning = c_det.get('judge_reasoning', 'N/A') or 'N/A'
    s_reasoning = s_det.get('judge_reasoning', 'N/A') or 'N/A'
    
    html = f"""
    <div style="border: 2px solid #e0e0e0; border-radius: 12px; margin: 16px 0; overflow: hidden; box-shadow: 0 2px 8px rgba(0,0,0,0.1); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <!-- Header -->
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 16px 20px;">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.8;">Task ID</div>
            <div style="font-size: 16px; font-weight: 600; font-family: monospace;">{task_id}</div>
        </div>
        
        <!-- Removed Segments -->
        <div style="padding: 16px 20px; background: #fafafa; border-bottom: 1px solid #e0e0e0;">
            <div style="font-size: 12px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Removed Segments</div>
            <div>{segments_html if segments_html else '<span style="color: #999;">None</span>'}</div>
        </div>
        
        <!-- Prompts Comparison -->
        <div style="display: grid; grid-template-columns: 1fr 1fr; border-bottom: 1px solid #e0e0e0;">
            <!-- Original Prompt -->
            <div style="padding: 16px 20px; border-right: 1px solid #e0e0e0;">
                <div style="font-size: 12px; color: #1976d2; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px; font-weight: 600;">Original Prompt</div>
                <div style="font-size: 14px; color: #333; line-height: 1.6; background: #f8f9fa; padding: 12px; border-radius: 6px; border-left: 3px solid #1976d2;">{original_prompt}</div>
            </div>
            <!-- Underspecified Prompt -->
            <div style="padding: 16px 20px;">
                <div style="font-size: 12px; color: #d32f2f; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px; font-weight: 600;">Underspecified Prompt</div>
                <div style="font-size: 14px; color: #333; line-height: 1.6; background: #fff8f8; padding: 12px; border-radius: 6px; border-left: 3px solid #d32f2f;">{underspec_prompt}</div>
            </div>
        </div>
        
        <!-- Expected Failure Mode -->
        <div style="padding: 16px 20px; border-bottom: 1px solid #e0e0e0;">
            <div style="font-size: 12px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Expected Failure Mode</div>
            <div style="font-size: 14px; color: #444; line-height: 1.5;">{c_det.get('expected_failure_mode', 'N/A')[:500]}{'...' if len(c_det.get('expected_failure_mode', '')) > 500 else ''}</div>
        </div>
        
        <!-- Method Comparison -->
        <div style="display: flex;">
            <!-- Completion Method -->
            <div style="flex: 1; padding: 16px 20px; background: {c_style['bg']}; border-right: 1px solid #e0e0e0;">
                <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 12px;">
                    <div style="font-size: 14px; font-weight: 600; color: #333;">Completion Method</div>
                    <span style="background: {c_style['badge']}; color: white; padding: 3px 10px; border-radius: 12px; font-size: 11px; text-transform: uppercase;">{c_det.get('category', 'N/A').replace('_', ' ')}</span>
                </div>
                <div style="margin-bottom: 8px;">
                    <span style="font-size: 12px; color: #666;">Verdict:</span>
                    <span style="font-size: 13px; font-weight: 500; margin-left: 4px;">{c_det.get('judge_verdict', 'N/A')}</span>
                </div>
                <div style="font-size: 13px; color: #555; line-height: 1.5; background: white; padding: 12px; border-radius: 8px; border: 1px solid {c_style['border']}20;">
                    {c_reasoning[:500]}{'...' if len(c_reasoning) > 500 else ''}
                </div>
            </div>
            
            <!-- Consistency Method -->
            <div style="flex: 1; padding: 16px 20px; background: {s_style['bg']};">
                <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 12px;">
                    <div style="font-size: 14px; font-weight: 600; color: #333;">Consistency Method</div>
                    <span style="background: {s_style['badge']}; color: white; padding: 3px 10px; border-radius: 12px; font-size: 11px; text-transform: uppercase;">{s_det.get('category', 'N/A').replace('_', ' ')}</span>
                </div>
                <div style="margin-bottom: 8px;">
                    <span style="font-size: 12px; color: #666;">Verdict:</span>
                    <span style="font-size: 13px; font-weight: 500; margin-left: 4px;">{s_det.get('judge_verdict', 'N/A')}</span>
                </div>
                <div style="font-size: 13px; color: #555; line-height: 1.5; background: white; padding: 12px; border-radius: 8px; border: 1px solid {s_style['border']}20;">
                    {s_reasoning[:500]}{'...' if len(s_reasoning) > 500 else ''}
                </div>
            </div>
        </div>
    </div>
    """
    display(HTML(html))

In [59]:
# Show tasks where Completion says "New" but Consistency says "Underspec"
display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 24px 0 16px 0;">
    <div style="display: flex; align-items: center; gap: 12px;">
        <div style="width: 8px; height: 32px; background: linear-gradient(180deg, #ff9800 0%, #f44336 100%); border-radius: 4px;"></div>
        <div>
            <h3 style="margin: 0; color: #333;">Completion → New Task, Consistency → Underspecified</h3>
            <p style="margin: 4px 0 0 0; color: #666; font-size: 14px;">{len(completion_new_consistency_underspec)} tasks where single response looked complete, but trials were inconsistent</p>
        </div>
    </div>
</div>
"""))

for task_id in list(completion_new_consistency_underspec)[:5]:  # Show first 5
    display_task(task_id, completion_results["task_details"], consistency_results["task_details"])

In [60]:
# Show tasks where Completion says "Underspec" but Consistency says "New"
display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 24px 0 16px 0;">
    <div style="display: flex; align-items: center; gap: 12px;">
        <div style="width: 8px; height: 32px; background: linear-gradient(180deg, #f44336 0%, #ff9800 100%); border-radius: 4px;"></div>
        <div>
            <h3 style="margin: 0; color: #333;">Completion → Underspecified, Consistency → New Task</h3>
            <p style="margin: 4px 0 0 0; color: #666; font-size: 14px;">{len(completion_underspec_consistency_new)} tasks where single response looked incomplete, but trials were actually consistent</p>
        </div>
    </div>
</div>
"""))

for task_id in list(completion_underspec_consistency_new)[:5]:  # Show first 5
    display_task(task_id, completion_results["task_details"], consistency_results["task_details"])

## Browse by Category

In [44]:
def create_task_browser(results, method_name):
    """Create a DataFrame for browsing tasks by category."""
    rows = []
    for task_id, details in results["task_details"].items():
        rows.append({
            "task_id": task_id,
            "category": details.get("category", "unknown"),
            "removed_segments": ", ".join(details.get("removed_segments", [])),
            "judge_verdict": details.get("judge_verdict", "N/A"),
            "underspec_n_success": details.get("underspec_n_success", 0),
            "judge_reasoning": (details.get("judge_reasoning", "") or "")[:150] + "...",
        })
    df = pd.DataFrame(rows)
    df["method"] = method_name
    return df

completion_df = create_task_browser(completion_results, "completion")
consistency_df = create_task_browser(consistency_results, "consistency")

In [45]:
# Browse originally overspecified tasks
print("=== Originally Overspecified Tasks (still pass@3 after underspec) ===")
display(completion_df[completion_df["category"] == "originally_overspecified"][["task_id", "removed_segments", "underspec_n_success"]].head(10))

=== Originally Overspecified Tasks (still pass@3 after underspec) ===


,task_id,removed_segments,underspec_n_success
0,6888e207a34beb25cfedda3b,average rating,1
1,6888e207a34beb25cfedda46,where this type of crime occurred most frequently,3
2,6866d4d2eea665b355a8f8ab,nvidia opening stock price,3
4,68925f01db31fe35560e3e85,first coined for automobiles,3
5,688ba1b3e95696e72dd93eaf,"title, authors and year",2
6,6880e844943a73a8c5e412c1,second bitcoin halving event,3
7,68925f01db31fe35560e3e9f,East Side between 77th and E 78th St NY,3
8,68925f01db31fe35560e3ea2,years elapsed between that publication and uru...,3
11,68925f01db31fe35560e3ebb,astronomy and horse trekking activities,2
12,688ba1b3e95696e72dd93e8d,guggenheim musuem in bilbao,1


In [46]:
# Browse new tasks (consistency method)
print("=== New Tasks - Consistency Method ===")
print("Tasks that failed pass@3 but had consistent responses across trials\n")
display(consistency_df[consistency_df["category"] == "new_task"][["task_id", "removed_segments", "judge_reasoning"]])

=== New Tasks - Consistency Method ===
Tasks that failed pass@3 but had consistent responses across trials



,task_id,removed_segments,judge_reasoning
13,688ba1b3e95696e72dd93e9c,average number of distinct individuals,All three trials correctly identified December...
24,6896416f7b30e5d8ccd7c8bf,mother's name,All three trials provide identical core inform...
29,689a373e148a49f91233dff7,talk directly with a model from terminal,All three trials provided the same substantive...
34,68940d8355faa0b720f0b4b4,the reformatory by tananarive due,All three trials identified July 2015 as the t...
35,689af4e653c3905e7b5b25c2,eight years after ac milan won their last cham...,All three trials identify Supplier_2 as the su...
36,689af4e653c3905e7b5b258e,largest national park in terms of acreage,All three trials identify the same southernmos...
42,68993ef3cf3e953b8ab83fd2,max verstappen became formula one world champi...,All three trials reached the same conclusion r...
50,689e0b1d9c8e2ac413c1f202,fourth commit after the initial commit,All three trials identified the same repositor...
52,689e0b1d9c8e2ac413c1f211,annualized rate of return,All three trials identified the exact same pub...
54,689e0b1d9c8e2ac413c1f21b,second commit,All three trials identified the exact same com...


In [47]:
# Browse underspecified tasks (consistency method)
print("=== Truly Underspecified Tasks - Consistency Method ===")
print("Tasks that failed pass@3 and had inconsistent responses across trials\n")
display(consistency_df[consistency_df["category"] == "underspecified"][["task_id", "removed_segments", "judge_reasoning"]])

=== Truly Underspecified Tasks - Consistency Method ===
Tasks that failed pass@3 and had inconsistent responses across trials



,task_id,removed_segments,judge_reasoning
3,6887c6d4614f06dc73af2411,koko.com,The trials are completely inconsistent. Trial ...
9,68925f01db31fe35560e3e97,SpaceX launched the first car into space,The trials are completely inconsistent because...
10,68925f01db31fe35560e3e8a,la voragine,The trials diverge significantly because they ...
14,68925f01db31fe35560e3ec1,wings gaming won their first the international...,The three trials provide completely different ...
15,6896416f7b30e5d8ccd7c8b4,the rehearsal of the ballet onstage,"Trials 1 and 3 are highly consistent, providin..."
21,68925f01db31fe35560e3ec2,vermeer tronie painted in 1665,The trials are highly inconsistent. Trial 2 fa...
22,68993ef3cf3e953b8ab83fa9,top movies file,The trials interpret the ambiguous term 'frequ...
28,689a373e148a49f91233dfce,head of the old hance trail,Trials 1 and 3 are consistent in identifying t...
41,689af4e653c3905e7b5b25a5,bob saget died,"Trial 1 reports a total of 3,586 visits, where..."
53,689bd255c0422b257e7dfcbe,2012 uefa champions league final,The trials diverge on which UEFA Champions Lea...


## Segment Dimension Analysis

In [48]:
# Load the original task JSONs to get segment dimensions
json_dir = DATASET_DIR / "json"

def get_segment_info(task_id):
    """Load segment info from the task JSON."""
    json_path = json_dir / f"{task_id}.json"
    if json_path.exists():
        with open(json_path) as f:
            data = json.load(f)
        variant = data.get("variants", [{}])[0]
        removed = variant.get("removed_segments", [])
        return {
            "dimensions": [s.get("dimension", "unknown") for s in removed],
            "subdimensions": [s.get("subdimension", "unknown") for s in removed],
            "values": [s.get("value", "unknown") for s in removed],
        }
    return {"dimensions": [], "subdimensions": [], "values": []}

# Analyze by category
def analyze_dimensions_by_category(task_ids, category_name):
    dimensions = []
    subdimensions = []
    for task_id in task_ids:
        info = get_segment_info(task_id)
        dimensions.extend(info["dimensions"])
        subdimensions.extend(info["subdimensions"])
    
    print(f"\n=== {category_name} ({len(task_ids)} tasks) ===")
    print("Dimensions:", dict(Counter(dimensions)))
    print("Subdimensions:", dict(Counter(subdimensions)))

In [49]:
# Analyze dimensions for each category (using consistency method)
analyze_dimensions_by_category(consistency_results["originally_overspecified"], "Originally Overspecified")
analyze_dimensions_by_category(consistency_results["new_tasks"], "New Tasks")
analyze_dimensions_by_category(consistency_results["underspecified"], "Underspecified")


=== Originally Overspecified (41 tasks) ===
Dimensions: {'goal': 23, 'input': 11, 'constraint': 7}
Subdimensions: {'target': 10, 'content': 12, 'location': 2, 'selection': 4, 'identifier': 7, 'source': 2, 'format': 1, 'temporal': 3}

=== New Tasks (12 tasks) ===
Dimensions: {'goal': 5, 'input': 2, 'constraint': 4, 'context': 1}
Subdimensions: {'target': 3, 'content': 2, 'identifier': 2, 'temporal': 1, 'selection': 3, 'terminology': 1}

=== Underspecified (17 tasks) ===
Dimensions: {'input': 9, 'context': 4, 'constraint': 2, 'goal': 2}
Subdimensions: {'identifier': 7, 'terminology': 4, 'temporal': 1, 'target': 2, 'source': 1, 'selection': 1, 'location': 1}


## Deep Dive: View Full Task Details

In [50]:
def show_full_task(task_id):
    """Show complete details for a task including prompts with beautiful HTML styling."""
    # Load task JSON
    json_path = json_dir / f"{task_id}.json"
    if not json_path.exists():
        print(f"Task JSON not found: {task_id}")
        return
    
    with open(json_path) as f:
        task_data = json.load(f)
    
    task = task_data.get("task", {})
    variant = task_data.get("variants", [{}])[0]
    
    c_det = completion_results["task_details"].get(task_id, {})
    s_det = consistency_results["task_details"].get(task_id, {})
    
    c_style = get_category_style(c_det.get('category'))
    s_style = get_category_style(s_det.get('category'))
    
    # Format segments with black text
    segments_html = ""
    for seg in variant.get("removed_segments", []):
        dim_colors = {"goal": "#1976d2", "input": "#7b1fa2", "constraint": "#388e3c", "context": "#f57c00"}
        color = dim_colors.get(seg.get('dimension'), '#616161')
        segments_html += f'''
        <div style="display: flex; align-items: center; gap: 10px; margin: 8px 0;">
            <span style="background: {color}; color: white; padding: 3px 10px; border-radius: 4px; font-size: 11px; min-width: 100px; text-align: center; font-weight: 500;">{seg.get('dimension')}/{seg.get('subdimension')}</span>
            <span style="font-size: 14px; color: #111; font-weight: 500; background: #f5f5f5; padding: 6px 12px; border-radius: 4px; border: 1px solid #ddd;">{seg.get('value')}</span>
        </div>'''
    
    c_reasoning = c_det.get('judge_reasoning', 'N/A') or 'N/A'
    s_reasoning = s_det.get('judge_reasoning', 'N/A') or 'N/A'
    
    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1000px;">
        <!-- Header -->
        <div style="background: linear-gradient(135deg, #1a237e 0%, #4a148c 100%); color: white; padding: 24px; border-radius: 16px 16px 0 0;">
            <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 1.5px; opacity: 0.7; margin-bottom: 4px;">Task Details</div>
            <div style="font-size: 20px; font-weight: 600; font-family: monospace;">{task_id}</div>
            <div style="margin-top: 8px; font-size: 13px; opacity: 0.8;">Category: {task.get('category', 'N/A')}</div>
        </div>
        
        <div style="border: 2px solid #e0e0e0; border-top: none; border-radius: 0 0 16px 16px; overflow: hidden;">
            <!-- Original Prompt -->
            <div style="padding: 20px; border-bottom: 1px solid #e0e0e0;">
                <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 12px;">
                    <div style="width: 4px; height: 20px; background: #1976d2; border-radius: 2px;"></div>
                    <div style="font-size: 14px; font-weight: 600; color: #1976d2; text-transform: uppercase; letter-spacing: 0.5px;">Original Prompt</div>
                </div>
                <div style="background: #f8f9fa; padding: 16px; border-radius: 8px; font-size: 14px; line-height: 1.6; color: #333; border-left: 4px solid #1976d2;">
                    {task.get('prompt', 'N/A')}
                </div>
            </div>
            
            <!-- Underspecified Prompt -->
            <div style="padding: 20px; border-bottom: 1px solid #e0e0e0;">
                <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 12px;">
                    <div style="width: 4px; height: 20px; background: #d32f2f; border-radius: 2px;"></div>
                    <div style="font-size: 14px; font-weight: 600; color: #d32f2f; text-transform: uppercase; letter-spacing: 0.5px;">Underspecified Prompt</div>
                </div>
                <div style="background: #fff8f8; padding: 16px; border-radius: 8px; font-size: 14px; line-height: 1.6; color: #333; border-left: 4px solid #d32f2f;">
                    {variant.get('underspecified_prompt', 'N/A')}
                </div>
            </div>
            
            <!-- Removed Segments -->
            <div style="padding: 20px; border-bottom: 1px solid #e0e0e0; background: #fafafa;">
                <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 12px;">
                    <div style="width: 4px; height: 20px; background: #7b1fa2; border-radius: 2px;"></div>
                    <div style="font-size: 14px; font-weight: 600; color: #7b1fa2; text-transform: uppercase; letter-spacing: 0.5px;">Removed Segments</div>
                </div>
                {segments_html if segments_html else '<div style="color: #999; font-style: italic;">None</div>'}
            </div>
            
            <!-- Expected Failure Mode -->
            <div style="padding: 20px; border-bottom: 1px solid #e0e0e0;">
                <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 12px;">
                    <div style="width: 4px; height: 20px; background: #ff6f00; border-radius: 2px;"></div>
                    <div style="font-size: 14px; font-weight: 600; color: #ff6f00; text-transform: uppercase; letter-spacing: 0.5px;">Expected Failure Mode</div>
                </div>
                <div style="font-size: 14px; line-height: 1.6; color: #555;">
                    {variant.get('expected_failure_mode', 'N/A')}
                </div>
            </div>
            
            <!-- Categorization Results -->
            <div style="padding: 20px;">
                <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 16px;">
                    <div style="width: 4px; height: 20px; background: #333; border-radius: 2px;"></div>
                    <div style="font-size: 14px; font-weight: 600; color: #333; text-transform: uppercase; letter-spacing: 0.5px;">Categorization Results</div>
                </div>
                
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
                    <!-- Completion -->
                    <div style="background: {c_style['bg']}; border: 2px solid {c_style['border']}; border-radius: 12px; padding: 16px;">
                        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px;">
                            <div style="font-weight: 600; color: #333;">Completion Method</div>
                            <span style="background: {c_style['badge']}; color: white; padding: 4px 12px; border-radius: 16px; font-size: 12px;">{c_det.get('category', 'N/A').replace('_', ' ').title()}</span>
                        </div>
                        <div style="font-size: 13px; color: #666; margin-bottom: 8px;">Verdict: <strong>{c_det.get('judge_verdict', 'N/A')}</strong></div>
                        <div style="font-size: 13px; color: #555; line-height: 1.5; background: white; padding: 12px; border-radius: 8px;">
                            {c_reasoning}
                        </div>
                    </div>
                    
                    <!-- Consistency -->
                    <div style="background: {s_style['bg']}; border: 2px solid {s_style['border']}; border-radius: 12px; padding: 16px;">
                        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px;">
                            <div style="font-weight: 600; color: #333;">Consistency Method</div>
                            <span style="background: {s_style['badge']}; color: white; padding: 4px 12px; border-radius: 16px; font-size: 12px;">{s_det.get('category', 'N/A').replace('_', ' ').title()}</span>
                        </div>
                        <div style="font-size: 13px; color: #666; margin-bottom: 8px;">Verdict: <strong>{s_det.get('judge_verdict', 'N/A')}</strong></div>
                        <div style="font-size: 13px; color: #555; line-height: 1.5; background: white; padding: 12px; border-radius: 8px;">
                            {s_reasoning}
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>
    """
    display(HTML(html))

In [53]:
# Example: Show a specific task (change task_id as needed)
# Pick an underspecified task from the consistency method
# example_task = consistency_results["underspecified"][0] if consistency_results["underspecified"] else None
example_task = "6888e207a34beb25cfedda3b"
if example_task:
    show_full_task(example_task)

In [34]:
# Interactive task viewer - enter a task ID
# show_full_task("6887c6d4614f06dc73af2411")  # The koko.com example

## Export for Further Analysis

In [35]:
# Create a combined DataFrame with both methods' results
combined_rows = []
for task_id in completion_results["task_details"].keys():
    c_det = completion_results["task_details"][task_id]
    s_det = consistency_results["task_details"].get(task_id, {})
    seg_info = get_segment_info(task_id)
    
    combined_rows.append({
        "task_id": task_id,
        "completion_category": c_det.get("category"),
        "consistency_category": s_det.get("category"),
        "methods_agree": c_det.get("category") == s_det.get("category"),
        "removed_segments": ", ".join(c_det.get("removed_segments", [])),
        "removed_dimensions": ", ".join(seg_info["dimensions"]),
        "underspec_n_success": c_det.get("underspec_n_success", 0),
        "completion_verdict": c_det.get("judge_verdict"),
        "consistency_verdict": s_det.get("judge_verdict"),
    })

combined_df = pd.DataFrame(combined_rows)
print(f"Combined DataFrame shape: {combined_df.shape}")
display(combined_df.head(10))

Combined DataFrame shape: (70, 9)


,task_id,completion_category,consistency_category,methods_agree,removed_segments,removed_dimensions,underspec_n_success,completion_verdict,consistency_verdict
0,6888e207a34beb25cfedda3b,originally_overspecified,originally_overspecified,True,average rating,goal,1,None,None
1,6888e207a34beb25cfedda46,originally_overspecified,originally_overspecified,True,where this type of crime occurred most frequently,goal,3,None,None
2,6866d4d2eea665b355a8f8ab,originally_overspecified,originally_overspecified,True,nvidia opening stock price,goal,3,None,None
3,6887c6d4614f06dc73af2411,underspecified,underspecified,True,koko.com,input,0,incomplete,inconsistent
4,68925f01db31fe35560e3e85,originally_overspecified,originally_overspecified,True,first coined for automobiles,goal,3,None,None
5,688ba1b3e95696e72dd93eaf,originally_overspecified,originally_overspecified,True,"title, authors and year",goal,2,None,None
6,6880e844943a73a8c5e412c1,originally_overspecified,originally_overspecified,True,second bitcoin halving event,goal,3,None,None
7,68925f01db31fe35560e3e9f,originally_overspecified,originally_overspecified,True,East Side between 77th and E 78th St NY,input,3,None,None
8,68925f01db31fe35560e3ea2,originally_overspecified,originally_overspecified,True,years elapsed between that publication and uru...,goal,3,None,None
9,68925f01db31fe35560e3e97,new_task,underspecified,False,SpaceX launched the first car into space,context,0,complete,inconsistent


In [36]:
# Save combined results
# combined_df.to_csv(DATASET_DIR / "categorization_combined.csv", index=False)
# print(f"Saved to {DATASET_DIR / 'categorization_combined.csv'}")